# 01 — SI Preprocessing

Scans all MEA recordings under `data_root`, registers a single pipeline task
**`preprocessing`**, and runs SpikeInterface preprocessing on every well:

1. Unsigned → signed conversion (when needed)
2. Bandpass filter
3. Local common median reference (fallback to global if electrode locations are missing)
4. Save preprocessed recording as a **zarr** compressed file

Recording names (`rec0000`, `rec0001`, …) and well IDs are discovered automatically
from each `data.raw.h5` file via `DatasetManager` — no hardcoded values needed.

**Configuration** is managed by `ConfigManager`. On the first run, a `pipeline_config.json`
template is generated — edit it to set your paths and parameters, then re-run.

The pipeline cache makes re-runs safe: already-complete wells are skipped automatically.
If preprocessing parameters change, `is_task_complete()` detects the stale config.
Failed wells can be reset with `pipeline_mgr.refresh("preprocessing", ...)`.

In [ ]:
import sys
import traceback
import logging
from pathlib import Path

import pandas as pd

# Allow importing from the repo root when running from notebooks/
_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from dataset_manager import DatasetManager
from pipeline_manager import PipelineManager
from pipeline_manager.task_record import TaskStatus
from pipeline_tasks import PreprocessingTask
from config_manager import ConfigManager

logging.basicConfig(level=logging.WARNING)  # suppress SI/DatasetManager chatter
print("Imports OK")

## Import preprocessing task

`PreprocessingTask` lives in `pipeline_tasks.preprocessing` so the notebook and future
executors use the same implementation. `default_params()` provides class-level fallback
values; any key present in `pipeline_config.json` under `tasks.preprocessing` overrides
those defaults via `resolve_params()` inside `run()`.

In [ ]:
print(f"Task imported: {PreprocessingTask.task_name!r}  deps={PreprocessingTask.dependencies}")
PreprocessingTask.default_params()

## Config

On the **first run** this cell writes a `pipeline_config.json` template and stops.
Edit the file to fill in your paths and parameters, then re-run to load it.

The template is seeded from `PreprocessingTask.default_params()` so all knobs are
pre-populated — just overwrite the values you want to change.

In [ ]:
# ============================================================
# Only this path needs to be set by hand.
CONFIG_FILE = Path("../pipeline_config.json")
# ============================================================

cm = ConfigManager()
cm.register_task(PreprocessingTask)   # seeds template with default_params()

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Template written to {CONFIG_FILE}.\n"
        "Edit it — set data_root, analysis_root, and task parameters — "
        "then re-run this cell."
    )

cm.load(CONFIG_FILE)

DATA_ROOT     = Path(cm.get_global("data_root"))
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))

print(f"Config loaded from: {CONFIG_FILE}")
print(f"  data_root:     {DATA_ROOT}")
print(f"  analysis_root: {ANALYSIS_ROOT}")
print(f"  task params:   {cm.get_task_params(PreprocessingTask.task_name)}")

## 1. Scan recordings

In [ ]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.get_by("scan_type", "==", "Network")

print(f"Found {len(recordings)} Network recording(s)")

summary_rows = [
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "scan_type":     r.scan_type,
        "run_id":        r.run_id,
        "rec_names":     list(r.h5_recordings.keys()),
        "n_wells":       len(r.wells),
        "size_GB":       round(r.file_size / 1e9, 2),
    }
    for r in recordings
]
pd.DataFrame(summary_rows)

## 2. Register pipeline task and add wells

In [ ]:
TASK_NAME = PreprocessingTask.task_name

pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

# register_task raises ValueError on duplicate — safe to re-run
try:
    pipeline_mgr.register_task(PreprocessingTask)
    print(f"Registered task: {TASK_NAME!r}")
except ValueError as e:
    print(f"Task already registered ({e}) — continuing")

n_added = 0
n_skipped = 0
for rec in recordings:
    if not rec.h5_recordings:
        print(f"WARNING: no h5 structure found for {rec.cache_key!r} — skipping")
        n_skipped += 1
        continue
    for rec_name, well_ids in rec.h5_recordings.items():
        for well_id in well_ids:
            # Compound key encodes both dimensions; PipelineManager treats it as opaque.
            pipeline_mgr.add_well(rec.cache_key, f"{rec_name}/{well_id}")
            n_added += 1

print(f"\nWork units registered: {n_added}  |  Recordings skipped (no h5 structure): {n_skipped}")

## 3. Pipeline status overview

In [ ]:
def _status_table(mgr: PipelineManager, task_name: str) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        rec_name, well_id = entry.well_id.split("/", 1)
        task = entry.tasks.get(task_name)
        rows.append({
            "recording_key": entry.recording_key,
            "rec_name":      rec_name,
            "well_id":       well_id,
            "status":        task.status if task else "—",
            "config_fresh":  mgr.is_task_complete(WorkItem(entry.recording_key, entry.well_id, task_name))
                             if task and task.status == TaskStatus.COMPLETE else "—",
            "output_path":   str(task.output_path) if task and task.output_path else "",
            "error":         (task.error or "")[:120] if task else "",
        })
    return pd.DataFrame(rows)


df_status = _status_table(pipeline_mgr, TASK_NAME)
print(df_status["status"].value_counts().to_string())
df_status

## 4. Run preprocessing

Parameters are read from `pipeline_config.json` and merged with `PreprocessingTask.default_params()`.
The effective params are **snapshotted** into `TaskRecord.config` when each well starts — so
`is_task_complete()` will return `False` for any well whose frozen params no longer match the
current config (i.e., you changed a parameter). Use `pipeline_mgr.refresh(TASK_NAME)` to
reset stale wells for re-processing.

Already-complete wells with a **fresh** config are not returned by `get_next_task()` and are
skipped automatically.  
To retry failed entries without a full reset: re-run with `RETRY_FAILED = True`.  
To reset a specific entry: `pipeline_mgr.refresh(TASK_NAME, recording_key=..., well_id="rec0000/well000")`  
To reset everything: `pipeline_mgr.refresh(TASK_NAME)`

In [ ]:
RETRY_FAILED = False  # set True to also retry previously failed entries

task = PreprocessingTask()
params = cm.get_task_params(TASK_NAME)   # values from pipeline_config.json

# Build a lookup dict for fast recording resolution
_rec_lookup = {r.cache_key: r for r in recordings}

while True:
    work_items = pipeline_mgr.get_next_task(n=1, retry_failed=RETRY_FAILED)
    if not work_items:
        break

    item      = work_items[0]
    rec_entry = _rec_lookup[item.recording_key]
    rec_name, well_id = item.well_id.split("/", 1)

    print(f"\nProcessing: {item.recording_key} / {rec_name} / {well_id}")
    pipeline_mgr.update_status(item, TaskStatus.RUNNING)  # snapshots params into TaskRecord

    try:
        output_path = task.run(
            item.recording_key,
            item.well_id,
            dataset_mgr.get_path(rec_entry),
            params,
        )
        pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=output_path)
        print(f"  ✓  saved → {output_path}")
    except Exception as exc:
        pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
        traceback.print_exc()
        print(f"  ✗  failed: {exc}")

print("\nDone — no more pending tasks.")

## 5. Final status report

In [ ]:
df_final = _status_table(pipeline_mgr, TASK_NAME)

print("=== Summary ===")
print(df_final["status"].value_counts().to_string())

stale = df_final[(df_final["status"] == TaskStatus.COMPLETE) & (df_final["config_fresh"] == False)]
if not stale.empty:
    print(f"\n⚠️  {len(stale)} well(s) completed with outdated config — run pipeline_mgr.refresh(TASK_NAME) to reset.")

failed = df_final[df_final["status"] == TaskStatus.FAILED]
if not failed.empty:
    print("\n=== Failed entries ===")
    for _, row in failed.iterrows():
        print(f"  {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
        print(f"    error: {row['error']}")

df_final